# Project 8: Food Delivery Operations Analytics
Google Colab version. Run the cells from top to bottom (**Runtime > Run all**).

**Before you start:** have `08_food_delivery_randomized_expanded.csv` ready on your computer. Step 1 will ask you to upload it.

Uses `df.head()`, `if/elif` and loops. No `groupby` and no IQR.

In [1]:
import pandas as pd
import numpy as np
from google.colab import files


## 1. Load And Inspect

**Upload and read the file**

In [2]:
# Step A: upload the file (a file picker appears - click "Choose files")
uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file was uploaded. Run this cell again and choose the CSV file.")

file_name = list(uploaded.keys())[0]
print("Uploaded:", file_name)

# Step B: read the first 9 columns (works for both .csv and .xlsx)
if file_name.endswith(".csv"):
    df = pd.read_csv(file_name, usecols=range(9))
elif file_name.endswith(".xlsx"):
    df = pd.read_excel(file_name, usecols=range(9))
else:
    raise ValueError("Please upload a .csv or .xlsx file. You uploaded: " + file_name)

Uploaded: 08_food_delivery_randomized_expanded.csv


**Inspect the data**

In [3]:
print("Shape:", df.shape)
print(df.head())
df.info()
print(df.describe())

Shape: (1500, 9)
  Order_ID       City   Restaurant  ...  Order_Status  Payment_Mode  Order_Hour
0  FD00547      Noida  Pizza Point  ...     Delivered          Card          15
1  FD00984   Gurugram  Pizza Point  ...     Delivered          Card          19
2  FD00712  Faridabad    Spice Hub  ...     Cancelled          Cash          23
3  FD00638  Faridabad  Pizza Point  ...     Cancelled          Cash          10
4  FD00471  Faridabad  Urban Bites  ...     Delivered          Card          13

[5 rows x 9 columns]
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order_ID           1500 non-null   str    
 1   City               1500 non-null   str    
 2   Restaurant         1500 non-null   str    
 3   Order_Value        1500 non-null   float64
 4   Distance_km        1500 non-null   float64
 5   Delivery_Time_Min  1477 non-null   float64
 6

**Lists of unique values (used in later tasks)**

In [4]:
cities = df["City"].unique()
restaurants = df["Restaurant"].unique()
payments = df["Payment_Mode"].unique()
statuses = df["Order_Status"].unique()

## 2. Missing Delivery Times

**Find missing values**

In [5]:
print("\n2. MISSING DELIVERY TIMES")
print("Missing values per column:")
print(df.isna().sum())
missing_rows = df[df["Delivery_Time_Min"].isna()]
print("Rows with missing delivery time:", len(missing_rows))
print(missing_rows.head())


2. MISSING DELIVERY TIMES
Missing values per column:
Order_ID              0
City                  0
Restaurant            0
Order_Value           0
Distance_km           0
Delivery_Time_Min    23
Order_Status          0
Payment_Mode          0
Order_Hour            0
dtype: int64
Rows with missing delivery time: 23
    Order_ID      City   Restaurant  ...  Order_Status  Payment_Mode  Order_Hour
64   FD00058     Noida   Green Bowl  ...     Delivered          Cash           8
127  FD00806  Gurugram    Spice Hub  ...     Delivered           UPI          23
154  FD00210     Delhi   Green Bowl  ...     Delivered           UPI          23
193  FD01201     Delhi  Pizza Point  ...     Delivered           UPI          10
268  FD01228     Delhi      Cafe 24  ...     Delivered          Card          12

[5 rows x 9 columns]


**Handle missing values**

In [6]:
# Decision: fill with the median delivery time of the same city
# (cancelled orders are left empty because they were never delivered).
for city in cities:
    city_median = df[df["City"] == city]["Delivery_Time_Min"].median()
    mask = (df["City"] == city) & (df["Delivery_Time_Min"].isna()) & (df["Order_Status"] != "Cancelled")
    df.loc[mask, "Delivery_Time_Min"] = city_median
print("Missing after cleaning:", df["Delivery_Time_Min"].isna().sum())

Missing after cleaning: 0


## 3. Duplicate Order_Id

In [7]:
print("\n3. DUPLICATE ORDER_ID")
dup_count = df["Order_ID"].duplicated().sum()
print("Duplicate Order_IDs:", dup_count)
if dup_count > 0:
    df = df.drop_duplicates(subset="Order_ID", keep="first")
    print("Duplicates removed. New shape:", df.shape)
else:
    print("No duplicates found, nothing to remove.")


3. DUPLICATE ORDER_ID
Duplicate Order_IDs: 0
No duplicates found, nothing to remove.


## 4. Total Orders And Status Percentages

In [8]:
print("\n4. TOTAL ORDERS AND STATUS %")
print("Total orders:", len(df))
print((df["Order_Status"].value_counts(normalize=True) * 100).round(2))


4. TOTAL ORDERS AND STATUS %
Total orders: 1500
Order_Status
Delivered    81.47
Delayed       9.80
Cancelled     8.73
Name: proportion, dtype: float64


## 5. Average Delivery Time By City

In [9]:
print("\n5. AVERAGE DELIVERY TIME BY CITY")
city_avg = {}
for city in cities:
    city_avg[city] = round(df[df["City"] == city]["Delivery_Time_Min"].mean(), 2)
    print(city, ":", city_avg[city], "min")
slowest_city = max(city_avg, key=city_avg.get)


5. AVERAGE DELIVERY TIME BY CITY
Noida : 37.45 min
Gurugram : 36.42 min
Faridabad : 34.45 min
Delhi : 35.84 min


## 6. Top 5 Restaurants By Total Order Value

In [10]:
print("\n6. TOP 5 RESTAURANTS BY TOTAL ORDER VALUE")
rest_total = {}
for r in restaurants:
    rest_total[r] = round(df[df["Restaurant"] == r]["Order_Value"].sum(), 2)
top5 = pd.Series(rest_total).sort_values(ascending=False).head(5)
print(top5)


6. TOP 5 RESTAURANTS BY TOTAL ORDER VALUE
Pizza Point    154567.86
Green Bowl     144228.91
Urban Bites    137221.30
Spice Hub      125999.64
Cafe 24        117395.25
dtype: float64


## 7. Average Order Value By Restaurant

In [11]:
print("\n7. AVERAGE ORDER VALUE BY RESTAURANT")
rest_avg = {}
for r in restaurants:
    rest_avg[r] = round(df[df["Restaurant"] == r]["Order_Value"].mean(), 2)
print(pd.Series(rest_avg).sort_values(ascending=False))


7. AVERAGE ORDER VALUE BY RESTAURANT
Cafe 24        469.58
Urban Bites    458.93
Green Bowl     454.98
Spice Hub      446.81
Pizza Point    439.11
dtype: float64


## 8. Peak Ordering Hours

In [12]:
print("\n8. PEAK ORDERING HOURS")
hour_counts = df["Order_Hour"].value_counts().sort_index()
print(hour_counts)
print("Top 3 peak hours:")
print(hour_counts.sort_values(ascending=False).head(3))
peak_hour = hour_counts.idxmax()


8. PEAK ORDERING HOURS
Order_Hour
7      13
8      87
9     104
10    110
11     96
12     59
13     94
14     85
15    100
16     65
17     97
18    112
19     90
20    110
21     88
22    111
23     79
Name: count, dtype: int64
Top 3 peak hours:
Order_Hour
18    112
22    111
10    110
Name: count, dtype: int64


## 9. Delivery Time Across Order Status

In [13]:
print("\n9. DELIVERY TIME BY ORDER STATUS")
for s in statuses:
    times = df[df["Order_Status"] == s]["Delivery_Time_Min"]
    print(s, "| orders:", len(times), "| mean:", round(times.mean(), 2),
          "| median:", times.median())


9. DELIVERY TIME BY ORDER STATUS
Delivered | orders: 1222 | mean: 36.09 | median: 35.0
Cancelled | orders: 131 | mean: 35.86 | median: 38.0
Delayed | orders: 147 | mean: 34.8 | median: 35.0


## 10. Cancellation And Delay Rates By City

In [14]:
print("\n10. CANCELLATION AND DELAY RATE BY CITY")
cancel_rate = {}
for city in cities:
    city_df = df[df["City"] == city]
    total = len(city_df)
    cancelled = len(city_df[city_df["Order_Status"] == "Cancelled"])
    delayed = len(city_df[city_df["Order_Status"] == "Delayed"])
    cancel_rate[city] = round(cancelled / total * 100, 2)
    print(city, "| cancellation:", cancel_rate[city], "% | delay:",
          round(delayed / total * 100, 2), "%")
worst_cancel_city = max(cancel_rate, key=cancel_rate.get)


10. CANCELLATION AND DELAY RATE BY CITY
Noida | cancellation: 9.07 % | delay: 10.16 %
Gurugram | cancellation: 11.25 % | delay: 10.03 %
Faridabad | cancellation: 6.71 % | delay: 8.5 %
Delhi | cancellation: 8.61 % | delay: 10.83 %


## 11. Numpy Median And Standard Deviation

In [15]:
print("\n11. NUMPY MEDIAN AND STD")
times = df["Delivery_Time_Min"].dropna().to_numpy()
median_time = np.median(times)
mean_time = np.mean(times)
std_time = np.std(times, ddof=1)
print("Median:", median_time)
print("Standard deviation:", round(std_time, 2))


11. NUMPY MEDIAN AND STD
Median: 35.0
Standard deviation: 12.06


## 12. Unusually Long Deliveries (Mean + 2 Standard Deviations, No Iqr)

In [16]:
print("\n12. UNUSUALLY LONG DELIVERIES")
limit = mean_time + 2 * std_time
print("Limit (mean + 2 x std):", round(limit, 2), "min")
long_deliveries = df[df["Delivery_Time_Min"] > limit]
print("Unusually long deliveries:", len(long_deliveries))
print(long_deliveries.head())
print("Count by city:")
print(long_deliveries["City"].value_counts())


12. UNUSUALLY LONG DELIVERIES
Limit (mean + 2 x std): 60.07 min
Unusually long deliveries: 45
    Order_ID       City   Restaurant  ...  Order_Status  Payment_Mode  Order_Hour
6    FD00392      Delhi    Spice Hub  ...       Delayed           UPI           9
25   FD00643   Gurugram  Urban Bites  ...     Delivered           UPI          14
60   FD01063      Noida    Spice Hub  ...     Delivered           UPI          19
63   FD01431  Faridabad    Spice Hub  ...     Delivered          Card          11
106  FD00602      Noida  Urban Bites  ...     Delivered          Card          13

[5 rows x 9 columns]
Count by city:
City
Noida        19
Delhi        12
Gurugram      8
Faridabad     6
Name: count, dtype: int64


## 13. Delivery-Time Bands (If / Elif)

In [17]:
print("\n13. DELIVERY TIME BANDS")
def get_band(minutes):
    if pd.isna(minutes):
        return "Unknown"
    elif minutes <= 30:
        return "Fast"
    elif minutes <= 45:
        return "Standard"
    else:
        return "Delayed"

df["Delivery_Band"] = df["Delivery_Time_Min"].apply(get_band)
print("Fast: up to 30 min | Standard: 31-45 min | Delayed: over 45 min")
print(df[["Order_ID", "Delivery_Time_Min", "Delivery_Band"]].head())
print((df["Delivery_Band"].value_counts(normalize=True) * 100).round(2))


13. DELIVERY TIME BANDS
Fast: up to 30 min | Standard: 31-45 min | Delayed: over 45 min
  Order_ID  Delivery_Time_Min Delivery_Band
0  FD00547               35.0      Standard
1  FD00984               26.0          Fast
2  FD00712               11.0          Fast
3  FD00638               55.0       Delayed
4  FD00471               16.0          Fast
Delivery_Band
Standard    47.53
Fast        32.20
Delayed     20.27
Name: proportion, dtype: float64


## 14. Revenue By City And Payment Mode

In [18]:
print("\n14. REVENUE BY CITY AND PAYMENT MODE")
revenue = pd.DataFrame(index=cities, columns=payments, dtype=float)
for city in cities:
    for p in payments:
        subset = df[(df["City"] == city) & (df["Payment_Mode"] == p)]
        revenue.loc[city, p] = round(subset["Order_Value"].sum(), 2)
revenue["Total"] = revenue.sum(axis=1)
print(revenue)


14. REVENUE BY CITY AND PAYMENT MODE
               Card      Cash       UPI      Total
Noida      57858.64  49871.77  55427.49  163157.90
Gurugram   47937.13  60395.81  48988.78  157321.72
Faridabad  87952.52  66950.89  38231.96  193135.37
Delhi      57163.12  51448.53  57186.32  165797.97


## 15. Operational Recommendations

In [19]:
print("\n15. RECOMMENDATIONS")
print("1. Add riders and review routes in", slowest_city,
      "- it has the slowest average delivery time (", city_avg[slowest_city], "min).")
print("2. Increase rider and kitchen staff around", peak_hour,
      ":00, the busiest ordering hour.")
print("3. Investigate", worst_cancel_city, "- highest cancellation rate (",
      cancel_rate[worst_cancel_city], "%). Improve ETA accuracy and order confirmation.")
print("4. Work with high-value restaurants like", top5.index[0],
      "to speed up dispatch and protect revenue.")
print("5. Track deliveries over", round(limit, 1),
      "min and send proactive updates or compensation to affected customers.")


15. RECOMMENDATIONS
1. Add riders and review routes in Noida - it has the slowest average delivery time ( 37.45 min).
2. Increase rider and kitchen staff around 18 :00, the busiest ordering hour.
3. Investigate Gurugram - highest cancellation rate ( 11.25 %). Improve ETA accuracy and order confirmation.
4. Work with high-value restaurants like Pizza Point to speed up dispatch and protect revenue.
5. Track deliveries over 60.1 min and send proactive updates or compensation to affected customers.


## 16. File Handling (Save The Results To Files)

**(a) Save cleaned dataset**

In [20]:
print("\n16. FILE HANDLING")

# (a) Save the cleaned dataset (with the delivery bands) as a CSV file
df.to_csv("cleaned_food_delivery.csv", index=False)
print("Saved: cleaned_food_delivery.csv")


16. FILE HANDLING
Saved: cleaned_food_delivery.csv


**(b) Save city performance table**

In [21]:
# (b) Save the city performance table as a CSV file
city_rows = []
for city in cities:
    city_df = df[df["City"] == city]
    total = len(city_df)
    city_rows.append({
        "City": city,
        "Total_Orders": total,
        "Avg_Delivery_Time_Min": city_avg[city],
        "Cancellation_Rate_%": cancel_rate[city],
        "Delay_Rate_%": round(len(city_df[city_df["Order_Status"] == "Delayed"]) / total * 100, 2),
        "Revenue": round(city_df["Order_Value"].sum(), 2),
    })
pd.DataFrame(city_rows).to_csv("city_performance.csv", index=False)
print("Saved: city_performance.csv")

Saved: city_performance.csv


**(c) Write the text report**

In [22]:
# (c) Write a text report using open() in write mode ("w")
try:
    with open("analysis_report.txt", "w") as f:
        f.write("FOOD DELIVERY OPERATIONS ANALYTICS - SUMMARY REPORT\n")
        f.write("=" * 52 + "\n\n")
        f.write("Total orders: " + str(len(df)) + "\n")
        f.write("Median delivery time: " + str(median_time) + " min\n")
        f.write("Std deviation: " + str(round(std_time, 2)) + " min\n")
        f.write("Unusually long limit (mean + 2 std): " + str(round(limit, 2)) + " min\n")
        f.write("Unusually long deliveries: " + str(len(long_deliveries)) + "\n\n")

        f.write("Average delivery time by city:\n")
        for city in cities:
            f.write("  " + city + ": " + str(city_avg[city]) + " min\n")

        f.write("\nTop 5 restaurants by total order value:\n")
        for name in top5.index:
            f.write("  " + name + ": " + str(top5[name]) + "\n")

        f.write("\nTop 3 peak hours:\n")
        for hour, count in hour_counts.sort_values(ascending=False).head(3).items():
            f.write("  " + str(hour) + ":00 -> " + str(count) + " orders\n")

        f.write("\nRecommendations:\n")
        f.write("  1. Add riders and review routes in " + slowest_city + ".\n")
        f.write("  2. Increase staff around " + str(peak_hour) + ":00 (peak hour).\n")
        f.write("  3. Investigate cancellations in " + worst_cancel_city + ".\n")
        f.write("  4. Work with " + top5.index[0] + " to speed up dispatch.\n")
        f.write("  5. Track deliveries over " + str(round(limit, 1)) + " min and update customers.\n")
    print("Saved: analysis_report.txt")
except IOError as error:
    print("Could not write the report:", error)

Saved: analysis_report.txt


**(d) Append to a log file**

In [23]:
# (d) Append a line to a log file using append mode ("a")
with open("analysis_log.txt", "a") as log:
    log.write("Analysis run on " + str(len(df)) + " orders. Report and CSV files created.\n")
print("Appended to: analysis_log.txt")

Appended to: analysis_log.txt


**(e) Read the report back**

In [24]:
# (e) Read the report back using read mode ("r") to confirm it was saved
try:
    with open("analysis_report.txt", "r") as f:
        content = f.read()
    print("\n--- Contents of analysis_report.txt ---")
    print(content)
except FileNotFoundError:
    print("analysis_report.txt was not found.")


--- Contents of analysis_report.txt ---
FOOD DELIVERY OPERATIONS ANALYTICS - SUMMARY REPORT

Total orders: 1500
Median delivery time: 35.0 min
Std deviation: 12.06 min
Unusually long limit (mean + 2 std): 60.07 min
Unusually long deliveries: 45

Average delivery time by city:
  Noida: 37.45 min
  Gurugram: 36.42 min
  Faridabad: 34.45 min
  Delhi: 35.84 min

Top 5 restaurants by total order value:
  Pizza Point: 154567.86
  Green Bowl: 144228.91
  Urban Bites: 137221.3
  Spice Hub: 125999.64
  Cafe 24: 117395.25

Top 3 peak hours:
  18:00 -> 112 orders
  22:00 -> 111 orders
  10:00 -> 110 orders

Recommendations:
  1. Add riders and review routes in Noida.
  2. Increase staff around 18:00 (peak hour).
  3. Investigate cancellations in Gurugram.
  4. Work with Pizza Point to speed up dispatch.
  5. Track deliveries over 60.1 min and update customers.



**(f) Download the files to your computer**

In [25]:
# (f) Download the saved files to your computer (Colab-only)
for name in ["analysis_report.txt", "city_performance.csv",
             "cleaned_food_delivery.csv", "analysis_log.txt"]:
    files.download(name)